# SpaceRL — Versión 2: Q-learning con Wrappers sobre Taxi-v3

> **Proyecto:** SpaceRL  
> **Versión:** 2 — Entorno envuelto con Wrappers de Gymnasium  
> **Algoritmo:** Q-learning tabular (idéntico a V1)  
> **Autores:** Álvaro · Marta · Paula

---

## ¿Qué cambia respecto a la Versión 1?

| Elemento | V1 | V2 |
|---|---|---|
| Entorno base | Taxi-v3 | Taxi-v3 (sin modificar) |
| Algoritmo | Q-learning tabular | Q-learning tabular (mismo) |
| Q-table shape | (500, 6) | (500, 6) (sin cambios) |
| Recompensa | Nativa | Modificada por RewardWrapper |
| Observación | Entero opaco | Recodificado semánticamente |
| Acciones | Directas | Con capa de seguimiento y safety-lock |
| Métricas | 5 columnas | 12 columnas (7 adicionales de wrappers) |

---

## 0. Imports y configuración

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import (
    N_EPISODES, LEARNING_RATE, DISCOUNT_FACTOR,
    EPSILON_START, EPSILON_DECAY,
    N_EVAL_EPS, QTABLE_PATH, METRICS_CSV_PATH, EVAL_CSV_PATH,
    FIGURE_PATH, COMPARISON_CSV_PATH, V1_METRICS_CSV, V1_EVAL_CSV,
    REWARD_INVALID_ACTION_PENALTY, REWARD_STEP_PENALTY,
    REWARD_DELIVERY_BONUS, REWARD_EFFICIENCY_BONUS,
)
from agent import QLearningAgent
from environment import make_env_v2, env_info_v2
from training import run_training_v2
from evaluation import evaluate_agent_v2, build_training_figure, build_comparison_figure
from utils import (
    save_training_metrics_v2, load_training_metrics_v2,
    save_evaluation_metrics_v2, load_evaluation_metrics_v2,
    training_summary_v2, evaluation_summary_v2,
    load_v1_metrics, build_comparison_csv, comparison_summary,
)
from wrappers import decode_taxi_state, state_to_spacerl_description

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

## 1. Hiperparámetros

In [ ]:
config_df = pd.DataFrame([
    {'Parámetro': 'N_EPISODES',                    'Valor': N_EPISODES,                   'Categoría': 'Training'},
    {'Parámetro': 'LEARNING_RATE',                 'Valor': LEARNING_RATE,                'Categoría': 'Training'},
    {'Parámetro': 'DISCOUNT_FACTOR',               'Valor': DISCOUNT_FACTOR,              'Categoría': 'Training'},
    {'Parámetro': 'EPSILON_START',                 'Valor': EPSILON_START,                'Categoría': 'Training'},
    {'Parámetro': 'EPSILON_DECAY',                 'Valor': EPSILON_DECAY,                'Categoría': 'Training'},
    {'Parámetro': 'REWARD_STEP_PENALTY',           'Valor': REWARD_STEP_PENALTY,          'Categoría': 'RewardWrapper'},
    {'Parámetro': 'REWARD_INVALID_ACTION_PENALTY', 'Valor': REWARD_INVALID_ACTION_PENALTY,'Categoría': 'RewardWrapper'},
    {'Parámetro': 'REWARD_DELIVERY_BONUS',         'Valor': REWARD_DELIVERY_BONUS,        'Categoría': 'RewardWrapper'},
    {'Parámetro': 'REWARD_EFFICIENCY_BONUS',       'Valor': REWARD_EFFICIENCY_BONUS,      'Categoría': 'RewardWrapper'},
])
config_df.style.set_caption('Configuración del experimento V2')

## 2. Análisis de observaciones — reinterpretación SpaceRL

In [ ]:
sample_states = [0, 42, 100, 200, 328, 499]
descriptions = [state_to_spacerl_description(s) for s in sample_states]
pd.DataFrame(descriptions).style.set_caption('Interpretación SpaceRL de estados de Taxi-v3')

## 3. Entrenamiento

>  Si ya tienes `results/metrics/metrics_v2.csv`, salta esta celda y ve a la sección 4.

In [ ]:
env   = make_env_v2()
agent = QLearningAgent(env)
print(f'Entrenando durante {N_EPISODES:,} episodios...\n')
metrics = run_training_v2(env, agent)
env.close()
agent.save(QTABLE_PATH)
df_train = save_training_metrics_v2(metrics, csv_path=METRICS_CSV_PATH)
print('\n Entrenamiento V2 completado.')

## 4. Análisis de métricas

In [ ]:
df_train = load_training_metrics_v2(METRICS_CSV_PATH)
df_train.head(8)

In [ ]:
df_train.describe().round(3)

In [ ]:
summary_v2 = training_summary_v2(df_train, last_n=500)
display(summary_v2.style.set_caption('Resumen V2 — últimos 500 episodios'))

## 5. Visualización

In [ ]:
build_training_figure(df_train, save_path=FIGURE_PATH, show=True)

## 6. Evaluación

In [ ]:
env_eval = make_env_v2()
agent_eval = QLearningAgent(env_eval)
env_eval.close()
agent_eval.load(QTABLE_PATH)
results = evaluate_agent_v2(agent_eval, n_episodes=N_EVAL_EPS)
df_eval = save_evaluation_metrics_v2(
    results['rewards'], results['steps'], results['successes'],
    results['invalid_actions'], results['wrapper_penalty'],
    results['wrapper_bonus'], results['blocked_actions'],
    results['useful_actions_ratio'],
    csv_path=EVAL_CSV_PATH,
)
print('\n Evaluación completada.')

In [ ]:
summary_eval = evaluation_summary_v2(df_eval)
display(summary_eval.style.set_caption('Resumen de evaluación V2'))

## 7. Comparación V1 vs V2

In [ ]:
df_v1 = load_v1_metrics(V1_METRICS_CSV)
if df_v1 is not None:
    df_cmp = build_comparison_csv(df_v1, df_train, out_path=COMPARISON_CSV_PATH)
    display(comparison_summary(df_v1, df_train, last_n=500).style.set_caption('V1 vs V2'))
    cmp_fig = FIGURE_PATH.replace('training_metrics_v2', 'v1_vs_v2_comparison')
    build_comparison_figure(df_v1, df_train, save_path=cmp_fig, show=True)
else:
    print('[INFO] Ejecuta primero el entrenamiento de V1.')

## 8. Política aprendida

In [ ]:
import numpy as np
ACTION_NAMES = ['Sur', 'Norte', 'Este', 'Oeste', 'Recoger', 'Dejar']
qtable = agent_eval.qtable
print(f'Q-table: {qtable.shape} | Rango: [{qtable.min():.2f}, {qtable.max():.2f}]')

rows = []
for s in [0, 100, 200, 300, 400, 499]:
    desc = state_to_spacerl_description(s)
    row = {'Estado': s, 'Posición': f"({desc['vehicle_row']},{desc['vehicle_col']})",
           'Carga': desc['cargo_status'], 'Acción óptima': ACTION_NAMES[np.argmax(qtable[s])]}
    for a, name in enumerate(ACTION_NAMES):
        row[name] = round(qtable[s, a], 2)
    rows.append(row)
pd.DataFrame(rows).style.set_caption('Q-values con contexto SpaceRL').highlight_max(subset=ACTION_NAMES, axis=1, color='#c8e6c9')